Suppose `Y = X @ W` where `X.shape = [N, Di]`, `Y.shape = [N, Do]` and `W.shape = [Di, Do]`.

We have `gradW = X^T @ gradY` and `gradX = gradY @ W^T`

In [3]:
import torch

def torch_grad(x: torch.Tensor, w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute the gradients of the loss with respect to the input and weights.

    Args:
        x (torch.Tensor): Input tensor.
        w (torch.Tensor): Weights tensor.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: Gradients with respect to input and weights.
    """
    # Ensure that x and w require gradients
    x.requires_grad_(True)
    w.requires_grad_(True)

    # Compute the output
    output = torch.matmul(x, w)

    # Compute the loss (for example, mean squared error)
    target = torch.zeros_like(output)  # Dummy target for demonstration
    loss = torch.nn.functional.mse_loss(output, target)

    # Backpropagate to compute gradients
    loss.backward()

    # Return the gradients
    return x.grad, w.grad

@torch.no_grad()
def manual_grad(x: torch.Tensor, w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Manually compute the gradients of the loss with respect to the input and weights.

    Args:
        x (torch.Tensor): Input tensor.
        w (torch.Tensor): Weights tensor.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: Gradients with respect to input and weights.
    """
    # Compute the output
    output = torch.matmul(x, w)

    # Compute the loss (for example, mean squared error)
    target = torch.zeros_like(output)  # Dummy target for demonstration
    loss = torch.mean((output - target) ** 2)

    # Compute gradients manually
    grad_output = 2 * (output - target) / output.numel()  # Gradient of loss w.r.t. output
    grad_w = torch.matmul(x.t(), grad_output)  # Gradient of loss w.r.t. weights
    grad_x = torch.matmul(grad_output, w.t())  # Gradient of loss w.r.t. input

    return grad_x, grad_w

In [6]:
x = torch.randn(16, 64)
w = torch.randn(64, 32)
torch_grad_x, torch_grad_w = torch_grad(x.clone().detach(), w.clone().detach())
manual_grad_x, manual_grad_w = manual_grad(x.clone().detach(), w.clone().detach())
print(f"Torch Gradients: {torch_grad_x.shape}, {torch_grad_w.shape}")
print(torch_grad_x)
print(torch_grad_w)
print(f"\nManual Gradients: {manual_grad_x.shape}, {manual_grad_w.shape}")
print(manual_grad_x)
print(manual_grad_w)

Torch Gradients: torch.Size([16, 64]), torch.Size([64, 32])
tensor([[ 0.3671, -0.0155,  0.1787,  ..., -0.2206, -0.0053, -0.1641],
        [ 0.1464, -0.5674, -0.0814,  ...,  0.2967,  0.0213,  0.2625],
        [ 0.1249,  0.5256, -0.0275,  ...,  0.1042,  0.2945, -0.0931],
        ...,
        [ 0.0573,  0.2224,  0.0040,  ..., -0.2624, -0.0614, -0.1945],
        [ 0.0154,  0.0326,  0.0533,  ..., -0.1423,  0.1897,  0.0470],
        [-0.0809, -0.1232, -0.1071,  ...,  0.0838,  0.2419, -0.2258]])
tensor([[ 0.0830, -0.1589,  0.0219,  ...,  0.1714,  0.1108,  0.1212],
        [-0.0160,  0.0298,  0.1022,  ..., -0.3274, -0.2781, -0.1702],
        [ 0.1252,  0.0008, -0.0063,  ...,  0.1619, -0.0728, -0.1453],
        ...,
        [ 0.0036,  0.0919,  0.1935,  ..., -0.2862, -0.1532, -0.0590],
        [ 0.0680,  0.0376,  0.0593,  ...,  0.0378, -0.0994, -0.0070],
        [ 0.0033, -0.1234,  0.0325,  ..., -0.1208, -0.2551,  0.1318]])

Manual Gradients: torch.Size([16, 64]), torch.Size([64, 32])
tensor([[ 